# Notebook 03 · Entrenamiento de Modelos

Este notebook entrena los cuatro clasificadores sobre los datos procesados en el Notebook 02
y optimiza el umbral de clasificacion en el conjunto de **validacion**.

**Modelos entrenados:**
1. Regresion Logistica - baseline lineal interpretable
2. Random Forest - ensemble de 300 arboles de decision
3. XGBoost - gradient boosting para datos tabulares
4. MLP (PyTorch) - red neuronal con 3 capas ocultas, Dropout y Early Stopping manual

**Restriccion fundamental:** el conjunto de test queda completamente vedado en este notebook.
Los umbrales se optimizan sobre validacion y se guardan para su aplicacion en el Notebook 04.

## 01 · Carga de artefactos procesados

Los arrays X_train, X_val, X_test y los vectores y_* se cargan desde
outputs/processed/, generados por el Notebook 02. El fichero meta.json contiene
las dimensiones, los pesos de clase y pos_weight = 4.185 para la perdida de PyTorch.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # backend no-GUI, evita cuelgues en VS Code

import os
# Evita deadlock de MKL/OpenMP en CPU (critico en Windows con torch)
os.environ['OMP_NUM_THREADS']      = '1'
os.environ['MKL_NUM_THREADS']      = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import os, site as _site
# Windows: pre-registra el directorio de DLLs de torch antes de cualquier import
for _sp in _site.getsitepackages():
    _td = os.path.join(_sp, 'torch', 'lib')
    if os.path.isdir(_td) and hasattr(os, 'add_dll_directory'):
        os.add_dll_directory(_td)
        break

import sys, pathlib, json, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = pathlib.Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROCESSED  = ROOT / 'outputs' / 'processed'
MODELS_DIR = ROOT / 'outputs' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(PROCESSED / 'X_train.npy')
X_val   = np.load(PROCESSED / 'X_val.npy')
X_test  = np.load(PROCESSED / 'X_test.npy')
y_train = np.load(PROCESSED / 'y_train.npy')
y_val   = np.load(PROCESSED / 'y_val.npy')
y_test  = np.load(PROCESSED / 'y_test.npy')

with open(PROCESSED / 'meta.json') as f:
    meta = json.load(f)

POS_WEIGHT = meta['pos_weight']
print(f'Train : {X_train.shape}  positivos: {y_train.mean():.1%}')
print(f'Val   : {X_val.shape}   positivos: {y_val.mean():.1%}')
print(f'Test  : {X_test.shape}   positivos: {y_test.mean():.1%}')
print(f'pos_weight = {POS_WEIGHT}')

## 02 · Regresion Logistica (baseline lineal)

La **Regresion Logistica** es el punto de referencia interpretable del proyecto.
Con class_weight='balanced' el algoritmo escala automaticamente el peso de cada
muestra para compensar el desbalance 4.18:1, aumentando el Recall de la clase positiva.

Sus coeficientes permitiran en el Notebook 04 verificar que las variables geneticas
(mut_BRCA1) y clinicas (obesidad, hipertension) tienen el mayor peso, coherente
con los lifts observados en el EDA.

In [ ]:
from src.models import train_logistic_regression

print('Entrenando Regresion Logistica...')
lr_model = train_logistic_regression(X_train, y_train)
print('Regresion Logistica entrenada')

## 03 · Random Forest (ensemble de arboles)

El **Random Forest** entrena 300 arboles independientes sobre sub-muestras bootstrap
y promedia sus predicciones (bagging). class_weight='balanced' ajusta internamente
el criterio de impureza para que los positivos reciban mas peso.

Se espera que mut_BRCA1 y obesidad lideren el ranking de importancia de features,
coherente con la senal de 3.92x y 24 pp observada en el EDA.

In [ ]:
from src.models import train_random_forest

print('Entrenando Random Forest (300 arboles)...')
rf_model = train_random_forest(X_train, y_train)
print('Random Forest entrenado')

## 04 · XGBoost (gradient boosting)

**XGBoost** construye arboles de forma secuencial: cada arbol corrige los residuos
del anterior (boosting con gradiente). El parametro scale_pos_weight reproduce el
ratio de desbalance (~4.185) para que el gradiente pese proporcionalmente mas los positivos.

Suele ser el modelo con mejor AUC-ROC en clasificacion tabular con variables mixtas,
gracias a su capacidad de capturar interacciones no lineales entre features.

In [ ]:
from src.models import train_xgboost

print('Entrenando XGBoost (400 estimadores)...')
xgb_model = train_xgboost(X_train, y_train, scale_pos_weight=POS_WEIGHT)
print('XGBoost entrenado')

## 05 · MLP con PyTorch (red neuronal profunda)

La **red neuronal MLP** sigue la arquitectura siguiente:

| Capa     | Dimension | Activacion | Regularizacion           |
|----------|-----------|------------|--------------------------|
| Entrada  | 19        | --         | --                       |
| Oculta 1 | 256       | ReLU       | BatchNorm + Dropout(0.3) |
| Oculta 2 | 128       | ReLU       | BatchNorm + Dropout(0.3) |
| Oculta 3 | 64        | ReLU       | BatchNorm + Dropout(0.3) |
| Salida   | 1         | --*        | --                       |

*La sigmoide se aplica implicitamente en BCEWithLogitsLoss al entrenar;
en inferencia se aplica explicitamente via 	orch.sigmoid.

**Mecanismos anti-sobreajuste:**
- BCEWithLogitsLoss(pos_weight=4.185) para el desbalance de clase
- Dropout 30% en cada capa oculta
- BatchNorm estabiliza los gradientes y acelera la convergencia
- **Early Stopping manual**: si al_loss no mejora en 12 epocas consecutivas, se restauran los mejores pesos
- weight_decay=1e-4 en Adam (regularizacion L2 implicita)

In [ ]:
from src.models import train_mlp

print('Entrenando MLP (PyTorch) con Early Stopping patience=25...')
mlp_model, history = train_mlp(
    X_train, y_train, X_val, y_val,
    pos_weight  = POS_WEIGHT,
    hidden_dims = (256, 128, 64),
    dropout     = 0.3,
    lr          = 1e-3,
    batch_size  = 2048,
    max_epochs  = 200,
    patience    = 25,
)

n_trained = len(history['train_loss'])
print(f'MLP entrenado: {n_trained} epocas totales')

## 06 · Optimizacion de umbral en validacion

El umbral por defecto de 0.5 es suboptimo con clases desbalanceadas: infradetecta
positivos innecesariamente. Para cada modelo se barre el rango [0.05, 0.95] en pasos
de 0.005 y se elige el umbral que **maximiza F1 en el conjunto de validacion**.

Este umbral queda fijado y se aplicara sin modificacion en el Notebook 04 sobre el
conjunto de test. En ningun caso el test participa en la seleccion del umbral.

In [ ]:
from src.evaluate import optimize_threshold

models = {
    'LR'     : lr_model,
    'RF'     : rf_model,
    'XGBoost': xgb_model,
    'MLP'    : mlp_model,
}

thresholds = {}
print('{:<10}  {:>8}  {:>8}'.format('Modelo', 'Umbral', 'F1-val'))
print('-' * 32)
for name, model in models.items():
    t, f1 = optimize_threshold(model, X_val, y_val)
    thresholds[name] = t
    print(f'{name:<10}  {t:>8.3f}  {f1:>8.4f}')

## 07 · Metricas en validacion (confirmacion)

Con el umbral optimo de cada modelo se evaluan las cinco metricas clave sobre validacion.
Esta tabla es unicamente una **comprobacion de sanidad** - las metricas oficiales del
proyecto se calcularan sobre el conjunto de test en el Notebook 04.

In [ ]:
import pandas as pd
from src.evaluate import eval_model

rows = []
for name, model in models.items():
    m = eval_model(model, X_val, y_val, thresholds[name])
    rows.append({'Modelo': name, **m})

df_val = (
    pd.DataFrame(rows)
    .set_index('Modelo')
    [['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']]
    .round(4)
)
print('Metricas en VALIDACION  (umbral optimizado - solo referencia, no son las metricas finales)')
print()
print(df_val.to_string())

## 08 · Guardado de modelos y umbrales

Los tres modelos de scikit-learn se serializan con pickle. El MLP de PyTorch se guarda
como state_dict (solo los pesos) junto a mlp_config.json con la arquitectura,
de forma que el Notebook 04 pueda reconstruirlo exactamente.

Los umbrales optimos se almacenan en 	hresholds.json para aplicarlos sobre test.

In [ ]:
import pickle
import torch

for tag, m in [('lr', lr_model), ('rf', rf_model), ('xgb', xgb_model)]:
    with open(MODELS_DIR / f'{tag}_model.pkl', 'wb') as f:
        pickle.dump(m, f)

torch.save(mlp_model.state_dict(), MODELS_DIR / 'mlp_weights.pt')

mlp_config = {'input_dim': int(X_train.shape[1]), 'hidden_dims': [256, 128, 64], 'dropout': 0.3}
with open(MODELS_DIR / 'mlp_config.json', 'w') as f:
    json.dump(mlp_config, f, indent=2)

with open(MODELS_DIR / 'thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)

print('Modelos guardados en outputs/models/:')
for p in sorted(MODELS_DIR.iterdir()):
    print(f'  {p.name:<30}  {p.stat().st_size / 1024:>8.1f} KB')